# 08.03 · RAG with frameworks: LangChain vs LlamaIndex vs Haystack

**Chapter:** Retrieval-Augmented Generation · **Milestone:** M3

## Goals
- [ ] Rebuild 08.01's pipeline in LangChain, LlamaIndex and Haystack, all on the same local model and embeddings
- [ ] See what each framework abstracts, and what that costs in debuggability
- [ ] Decide when to use a framework vs plain code

## Resources
- [LangChain – RAG tutorial](https://python.langchain.com/docs/tutorials/rag/) — free
- [LlamaIndex – Starter tutorial](https://docs.llamaindex.ai/en/stable/getting_started/starter_example/) — free
- [Haystack – Tutorials](https://haystack.deepset.ai/tutorials) — free

| Framework | Personality | Strong at |
|---|---|---|
| **LangChain** (+ LangGraph) | huge ecosystem of integrations; composable runnables | gluing many providers/tools; agents via LangGraph |
| **LlamaIndex** | data framework: loaders, indexes, query engines | document-heavy RAG, many retrieval strategies out of the box |
| **Haystack** | explicit pipelines of typed components | production pipelines that are easy to inspect and serialise |
| **Plain code** (08.01) | you own every line | small pipelines, full control, easiest debugging |

All three use the same pieces: a 500-passage subset, bge-small embeddings, and the `aieng.llm` provider via its OpenAI-compatible endpoint.

In [ ]:
import time

from aieng import rag
from aieng.llm import client

c, MODEL = client()
BASE_URL, API_KEY = str(c.base_url), c.api_key
LOCAL = c.base_url.host == "localhost"
NO_THINK = {"reasoning_effort": "none"} if LOCAL else {}

docs = rag.corpus()[:500]  # subset: frameworks re-embed everything, keep it fast
QUESTION = "How old was Lincoln when his mother died?"
print(MODEL, BASE_URL)

## 1. LangChain
Pieces: an embeddings object, a vector store, a retriever, a chat model, a prompt template; composed with `|` (LCEL).

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

start = time.perf_counter()
lc_emb = HuggingFaceEmbeddings(model_name=rag.EMBED_MODEL, encode_kwargs={"normalize_embeddings": True})
store = InMemoryVectorStore.from_texts(docs, lc_emb)
retriever = store.as_retriever(search_kwargs={"k": 5})
llm = ChatOpenAI(model=MODEL, base_url=BASE_URL, api_key=API_KEY, temperature=0, **NO_THINK)
prompt = ChatPromptTemplate.from_template(rag.PROMPT)

chain = (
    {"sources": retriever | (lambda ds: "\n\n".join(f"[{i}] {d.page_content}" for i, d in enumerate(ds))),
     "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)
print(chain.invoke(QUESTION), f"\n({time.perf_counter() - start:.1f}s incl. indexing)")

## 2. LlamaIndex
Pieces: `Document`s → `VectorStoreIndex` (chunks + embeds for you) → `query_engine` (retrieve + synthesise with its own prompt).

In [ ]:
from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai_like import OpenAILike

start = time.perf_counter()
Settings.embed_model = HuggingFaceEmbedding(model_name=rag.EMBED_MODEL)
Settings.llm = OpenAILike(model=MODEL, api_base=BASE_URL, api_key=API_KEY, is_chat_model=True, temperature=0,
                          additional_kwargs=NO_THINK, context_window=32_000)
li_index = VectorStoreIndex.from_documents([Document(text=d) for d in docs])
query_engine = li_index.as_query_engine(similarity_top_k=5)
resp = query_engine.query(QUESTION)
print(resp.response, f"\n({time.perf_counter() - start:.1f}s incl. indexing)")
print("sources:", [round(n.score, 3) for n in resp.source_nodes])

## 3. Haystack
Pieces: components (embedder, retriever, prompt builder, generator) connected explicitly in a `Pipeline` graph.

In [ ]:
from haystack import Document as HDocument, Pipeline, component
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.dataclasses import ChatMessage
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.utils import Secret


@component  # a custom component: any class with a typed run() method. Haystack 3 moved its sentence-transformers embedders to integrations.
class QueryEmbedder:
    @component.output_types(embedding=list[float])
    def run(self, text: str):
        return {"embedding": rag.embed([text], query=True)[0].tolist()}


start = time.perf_counter()
doc_store = InMemoryDocumentStore()
doc_store.write_documents([HDocument(content=d, embedding=v.tolist()) for d, v in zip(docs, rag.index()[:500])])  # reuse cached vectors

template = [ChatMessage.from_user(rag.PROMPT.replace("{sources}", "{% for d in documents %}[{{ loop.index0 }}] {{ d.content }}\n\n{% endfor %}")
                                  .replace("{question}", "{{ question }}"))]
pipe = Pipeline()
pipe.add_component("embed_query", QueryEmbedder())
pipe.add_component("retriever", InMemoryEmbeddingRetriever(doc_store, top_k=5))
pipe.add_component("prompt", ChatPromptBuilder(template=template, required_variables=["question", "documents"]))
pipe.add_component("llm", OpenAIChatGenerator(model=MODEL, api_base_url=BASE_URL, api_key=Secret.from_token(API_KEY),
                                              generation_kwargs={"temperature": 0, **NO_THINK}))
pipe.connect("embed_query.embedding", "retriever.query_embedding")
pipe.connect("retriever.documents", "prompt.documents")
pipe.connect("prompt.prompt", "llm.messages")

out = pipe.run({"embed_query": {"text": QUESTION}, "prompt": {"question": QUESTION}})
print(out["llm"]["replies"][0].text, f"\n({time.perf_counter() - start:.1f}s incl. indexing)")

## 4. Plain code, for comparison
08.01 in four lines (same 3,200 passages, not just 500):

In [ ]:
start = time.perf_counter()
text, hits = rag.answer(QUESTION)
print(text, f"\n({time.perf_counter() - start:.1f}s, index cached)")

## 5. When to use what
- **Frameworks pay off** when you need many integrations (loaders for Notion/Drive/Slack, 20 vector stores, tracing hooks), standard
  advanced patterns quickly (LlamaIndex's retrievers, LangGraph's agents), or a team convention.
- **They cost** layers of abstraction, hidden prompts (LlamaIndex's synthesis prompt isn't yours unless you override it), version churn,
  and harder debugging. Print the final prompt in each framework before you trust it.
- Common pattern: plain code for the core loop, framework pieces where they save real work (a loader, a splitter, a retriever).

## Exercises
1. Find and print the exact prompt LlamaIndex sent to the model (hint: `Settings.callback_manager` / `llama_index.core.set_global_handler("simple")`).
2. Swap the vector store in each framework for Chroma or Qdrant (07.03). How many lines change?
3. Add hybrid search (BM25 + vectors) in one framework of your choice. Compare effort with your 07.04 implementation.

In [ ]:
# your code here